# Orquestador de segmentación transaccional (BigQuery)
El notebook **solo orquesta**: toda la lógica vive en `sql/*.sql`, parametrizada.

| Modo | Pasos |
|---|---|
| `FULL` | 01 base → 02 resumen → 03 asignación por percentil → 04 umbrales |
| `NEXT_MONTH` | 01 base → 01b cambios de boca → 02 resumen → 03 incremental (reevalúa "Sin asignar" + clientes con cambio de boca relevante + nuevos) |
| `REASSIGN` | 01 base → 01b cambios de boca → 02 resumen → 03 incremental (reevalúa todos + nuevos) |

`NEXT_MONTH` y `REASSIGN` requieren una segmentación previa (`p_corte_prev`) y umbrales de un FULL (`p_corte_umbrales`) ya cargados.
Ejecutable con papermill: la celda siguiente lleva el tag `parameters`.

**Segmentación previa oficial + tabla alterna:** con `fuente_prev = "oficial"` el corte previo se carga desde `cr_segmentacion` (paso 00c). Con `exportar_alterna = True` el paso 05 genera la tabla en formato `temp_estrategica_transaccional` con la segmentación transaccional corregida.

**Boca de salida:** con `boca_desde_historico = True` la boca de cada cliente es la de `cr_cliente_dex_historico` en el último mes de la ventana (`p_corte - 1 mes`), con respaldo en su última venta. El paso 01b registra los cambios de boca vs. el corte previo en `seg_cambios_boca` (`CAMBIO_UNIVERSO` CMP↔AS↔OTRO, `CAMBIO_INTERNO`, `NUEVO`, `SIN_ACTUAL`). En `NEXT_MONTH` se reevalúan con los umbrales de su boca nueva los `CAMBIO_UNIVERSO` y los `CAMBIO_INTERNO` cuyo grupo vs2 cambia y sus umbrales difieren. En la tabla alterna, `cod_boca_salida` se actualiza con la boca del histórico. El paso 06 muestra el resumen de cambios por universo (QA).

**Diamante** (solo `FULL` y `REASSIGN`): los grupos de `grupos_diamante` se agrupan como uno solo (por SAP) y los clientes con percentil de UB acumulada ≤ `p_diamante` pasan a `0 Diamante`, con prioridad sobre su segmentación regular. `NEXT_MONTH` no evalúa Diamante: usa solo los umbrales de Oro+/Oro/Plata/Bronce y conserva los Diamante previos.

In [1]:
# ── PARÁMETROS ──────────────────────────────────────────────────────────────
modo = "NEXT_MONTH"                  # FULL | NEXT_MONTH | REASSIGN
p_corte = "2026-09-01"         # 1er día del mes que NO entra; ventana = 12 meses previos
p_corte_prev = "2026-08-01"              # NEXT_MONTH/REASSIGN: corte de la segmentación previa
p_corte_umbrales = "2026-01-01"          # NEXT_MONTH/REASSIGN: corte del FULL cuyos umbrales se congelan

project = "acpe-dev-uc-sandbox-aa"
dataset = "dev_segmentacion"
sql_dir = "sql"

# Fuentes (cambiar aquí para dev/prod)
src_kpi        = "acpe-prod-dlk-shrd-gld.delivery_contabilidad_gestion.cr_contabilidad_gestion_kpi"
src_sellout    = "acpe-prod-dlk-shrd-gld.delivery_reporte_comercial.cr_venta_sell_out"
src_cliente_dex_hist = "acpe-prod-dlk-shrd-gld.delivery_modelo_dimensional.cr_cliente_dex_historico"
src_cliente_alicorp  = "acpe-prod-dlk-shrd-gld.delivery_modelo_dimensional.cr_cliente_alicorp"
src_cliente    = "acpe-prod-dlk-shrd-gld.gld_cliente.dex_cliente"
src_odoo       = "acpe-prod-dlk-shrd-gld.gld_cliente.odoo_mk_dex_cliente"
src_estrategica_fuente = "acpe-prod-uc-ml.delivery_gtm.dex_b2b_cmp_segmentacion_estrategica"
                               # fec_corrida, id_interlocutor_origen, des_segmento_estrategico, cod_boca_salida
                               # los golosineros se copian por corte a {ds}.seg_estrategica (paso 00b)
permitir_sin_golosineros = False  # False: si no hay golosineros para el corte, el pipeline se detiene

# Diamante (FULL y REASSIGN)
p_diamante = 0.10              # percentil de UB acumulada (grupo agrupado, por SAP) para 0 Diamante
grupos_diamante = ["PA", "LV", "GA"]   # grupos tratados como uno solo

# Segmentación previa oficial y tabla alterna
fuente_prev = "oficial"        # "interna": seg_segmentacion del corte previo | "oficial": cr_segmentacion (solo NEXT_MONTH/REASSIGN)
src_prev_oficial = "acpe-prod-dlk-shrd-gld.delivery_cliente_gtm.cr_segmentacion"
exportar_alterna = True       # True: paso 05, genera la tabla alterna para p_corte
out_tabla_alterna = None       # None -> {project}.{dataset}.seg_tabla_alterna_estrategica_transaccional
completar_con_oficial = True  # True: agrega a la alterna los clientes de la oficial del periodo que no estan en la tabla estrategica
boca_desde_historico = True    # True: la boca sale de cr_cliente_dex_historico (ultimo mes de la ventana); False: ultima venta (regla anterior)
derivar_boca_por_grupo = True # True: clientes nuevos sin boca en ninguna fuente se agregan con boca derivada de su grupo de precio
respetar_diamante_base = True # True: la lista de '0 Diamante' de la oficial (cr_segmentacion) del periodo manda en la alterna

# Control
dry_run = False                # True: solo estima bytes, no escribe datos
max_gb = 200                   # tope duro de bytes facturados por paso
min_match_vb = 0.80            # compuerta del paso 01: % de venta del output que encuentra ratio (calibrar)

In [2]:
import re, json, time, uuid, datetime as dt
from pathlib import Path
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=project)
ds = f"{project}.{dataset}"
out_tabla_alterna = out_tabla_alterna or f"{ds}.seg_tabla_alterna_estrategica_transaccional"
src_estrategica = f"{ds}.seg_estrategica"   # snapshot por corte (lo que lee el paso 01)

CTX = dict(ds=ds, src_kpi=src_kpi, src_sellout=src_sellout, src_cliente_dex_hist=src_cliente_dex_hist, src_cliente_alicorp=src_cliente_alicorp,
           src_cliente=src_cliente, src_odoo=src_odoo, src_estrategica=src_estrategica,
           src_estrategica_fuente=src_estrategica_fuente,
           src_prev_oficial=src_prev_oficial, out_tabla_alterna=out_tabla_alterna,
           grupos_diamante=", ".join(f"'{g}'" for g in grupos_diamante),
           prev_tabla=f"{ds}.seg_prev_oficial" if fuente_prev == "oficial" else f"{ds}.seg_segmentacion")

PIPELINES = {
    "FULL":       ["00b_snapshot_estrategica", "01_base_cliente_mes", "02_resumen_cliente", "03_asignacion_full", "04_umbrales"],
    "NEXT_MONTH": ["00b_snapshot_estrategica", "01_base_cliente_mes", "01b_cambios_boca", "02_resumen_cliente", "03_asignacion_incremental"],
    "REASSIGN":   ["00b_snapshot_estrategica", "01_base_cliente_mes", "01b_cambios_boca", "02_resumen_cliente", "03_asignacion_incremental"],
}
assert modo in PIPELINES, f"modo inválido: {modo}"
assert fuente_prev in ("interna", "oficial"), "fuente_prev debe ser 'interna' u 'oficial'"
assert not (fuente_prev == "oficial" and modo == "FULL"), "fuente_prev='oficial' solo aplica a NEXT_MONTH/REASSIGN"

STEPS = list(PIPELINES[modo])
if fuente_prev == "oficial":
    STEPS.insert(STEPS.index("03_asignacion_incremental"), "00c_cargar_previa_oficial")
if exportar_alterna:
    STEPS.append("05_export_tabla_alterna")

VALUES = {
    "p_corte": dt.date.fromisoformat(p_corte),
    "p_prev": dt.date.fromisoformat(p_corte_prev) if p_corte_prev else None,
    "p_umb": dt.date.fromisoformat(p_corte_umbrales) if p_corte_umbrales else None,
    "solo_sin_asignar": (modo == "NEXT_MONTH"),
    "p_diamante": float(p_diamante),
    "respetar_diamante_base": bool(respetar_diamante_base),
    "completar_con_oficial": bool(completar_con_oficial),
    "derivar_boca_por_grupo": bool(derivar_boca_por_grupo),
    "boca_desde_historico": bool(boca_desde_historico),
}
PTYPES = {"p_corte": "DATE", "p_prev": "DATE", "p_umb": "DATE", "solo_sin_asignar": "BOOL", "p_diamante": "FLOAT64", "respetar_diamante_base": "BOOL", "completar_con_oficial": "BOOL", "derivar_boca_por_grupo": "BOOL", "boca_desde_historico": "BOOL"}

def render(sql):
    def rep(m):
        k = m.group(1)
        if k not in CTX:
            raise KeyError(f"Variable de plantilla no definida: {k}")
        return str(CTX[k])
    return re.sub(r"\{\{\s*(\w+)\s*\}\}", rep, sql)

def qparams(sql):
    """Solo pasa los parámetros que el SQL realmente usa."""
    return [bigquery.ScalarQueryParameter(n, PTYPES[n], VALUES[n]) for n in sorted(set(re.findall(r"@(\w+)", sql)))]

def cfg(dry=False, sql=""):
    return bigquery.QueryJobConfig(
        query_parameters=qparams(sql), dry_run=dry, use_query_cache=not dry,
        maximum_bytes_billed=None if dry else int(max_gb * 1024**3),
        labels={"pipeline": "segmentacion", "modo": modo.lower()})

C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
# ── Validación de parámetros ────────────────────────────────────────────────
def n_filas(tabla, corte):
    q = f"SELECT COUNT(*) n FROM `{ds}.{tabla}` WHERE fecha_corte = @c"
    job = client.query(q, job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("c", "DATE", corte)]))
    return list(job.result())[0].n

if modo != "FULL":
    if not (p_corte_prev and p_corte_umbrales):
        raise ValueError(f"{modo} requiere p_corte_prev y p_corte_umbrales")
    # Con la previa oficial (tabla aparte) puede coincidir con p_corte: se reevalúa el mismo periodo.
    if VALUES["p_prev"] > VALUES["p_corte"] or (VALUES["p_prev"] == VALUES["p_corte"] and fuente_prev != "oficial"):
        raise ValueError("p_corte_prev debe ser anterior a p_corte (o igual, solo con fuente_prev='oficial')")
    # Las tablas pueden no existir aún: el DDL se corre primero (abajo). Aquí solo se valida si ya existen.
print(f"modo={modo} | p_corte={p_corte} | ventana={VALUES['p_corte'] - dt.timedelta(days=365)} → {p_corte} (12 meses)")
print("pasos:", " → ".join(STEPS))

modo=NEXT_MONTH | p_corte=2026-09-01 | ventana=2025-09-01 → 2026-09-01 (12 meses)
pasos: 00b_snapshot_estrategica → 01_base_cliente_mes → 02_resumen_cliente → 00c_cargar_previa_oficial → 03_asignacion_incremental → 05_export_tabla_alterna


In [4]:
# ── Compuertas de calidad (se evalúan sobre la fila de métricas que devuelve cada SQL) ──
def gate_00b(m):
    ok = permitir_sin_golosineros or m["n_golosineros"] > 0
    return ok, f"golosineros={m['n_golosineros']:,} para {p_corte}"
def gate_00c(m):
    return m["n_filas"] > 0, f"previa oficial: clientes={m['n_filas']:,}, por_definir={m['n_por_definir']:,}, diamante={m['n_diamante']:,}, dex_con_varios_segmentos={m['n_dex_multi_seg']:,}"
def gate_05(m):
    ok = m["n_filas"] > 0 and (not respetar_diamante_base or m["n_diamante_final"] == m["n_diamante_base"])
    return ok, (f"filas={m['n_filas']:,}, sin_segmentacion_interna={m['n_sin_segmentacion_interna']:,}, "
                f"difiere_de_base={m['n_difiere_de_base']:,}, por_definir={m['n_por_definir']:,}, seg_fuera_de_base={m['n_seg_fuera_de_base']:,}, "
                f"diamante base/final={m['n_diamante_base']:,}/{m['n_diamante_final']:,} (en filas de la estrategica: {m['n_diamante_base_estrategica']:,}), "
                f"filas_completadas_con_oficial={m['n_filas_de_oficial']:,}, filas_boca_derivada={m['n_filas_derivadas']:,}, boca_actualizada_con_historico={m['n_boca_actualizada']:,}, sin_segmento_final={m['n_sin_segmento_final']:,}, "
                f"diamante_base_pisa_nuestro={m['n_diamante_base_pisa_nuestro']:,}, nuestro_fuera_de_lista={m['n_diamante_nuestro_fuera_de_lista']:,}")
def gate_01(m):
    ok = m["n_clientes"] > 0 and (m["pct_match_vb"] or 0) >= min_match_vb
    return ok, f"pct_match_vb={m['pct_match_vb']:.3f} (mín {min_match_vb}), n_clientes={m['n_clientes']:,}, dex_con_varios_sap={m['n_dex_multi_sap']:,}, sap_distinto_dim={m['n_dex_sap_dif_dim']:,}, sin_historico={m['n_dex_sin_hist']:,}, boca_hist_dif_ult_venta={m['n_dex_boca_hist_dif_venta']:,}"
def gate_01b(m):
    # 01b no bloquea por conteos de cambios: solo exige datos en el historico en ambos meses
    ok = m["n_filas"] > 0 and (m["n_sin_cambio"] + m["n_cambio_interno"] + m["n_cambio_universo"]) > 0
    return ok, (f"meses historico {m['mes_hist_ant']}→{m['mes_hist_act']}: clientes={m['n_filas']:,}, sin_cambio={m['n_sin_cambio']:,}, "
                f"cambio_interno={m['n_cambio_interno']:,}, cambio_universo={m['n_cambio_universo']:,}, nuevos={m['n_nuevo']:,}, sin_actual={m['n_sin_actual']:,}")
def gate_02(m):
    return m["n_clientes"] > 0, f"n_clientes={m['n_clientes']:,}, sin_asignar={m['n_sin_asignar']:,}, sin_sap={m['n_sin_sap']:,}"
def gate_03(m):
    ok = m["n_filas"] > 0 and m["n_sin_segmento"] == 0 and m["n_filas"] >= (m.get("n_filas_prev") or 0)
    return ok, f"n_filas={m['n_filas']:,}, sin_segmento={m['n_sin_segmento']}, filas_prev={m.get('n_filas_prev')}, diamante={m.get('n_diamante')}, sin_asignar={m.get('n_sin_asignar_seg')}, reeval_cambio_boca={m.get('n_reevaluado_cambio_boca')}, reeval_sin_umbral={m.get('n_reeval_sin_umbral')}"
def gate_04(m):
    return m["n_filas"] > 0, f"umbrales={m['n_filas']:,}, umbrales_diamante={m['n_umbrales_diamante']:,}"
GATES = {"00b_snapshot_estrategica": gate_00b, "00c_cargar_previa_oficial": gate_00c, "05_export_tabla_alterna": gate_05, "01_base_cliente_mes": gate_01, "01b_cambios_boca": gate_01b, "02_resumen_cliente": gate_02,
         "03_asignacion_full": gate_03, "03_asignacion_incremental": gate_03, "04_umbrales": gate_04}

RUN_ID = uuid.uuid4().hex[:12]
RESULTS = []

def log(step, status, bytes_billed=None, secs=None, metrics=None, error=None):
    row = {"run_id": RUN_ID, "ts": dt.datetime.utcnow().isoformat(), "modo": modo, "p_corte": p_corte,
           "step": step, "status": status, "bytes_billed": bytes_billed, "seconds": secs,
           "metrics": json.dumps(metrics, default=str) if metrics else None, "error": error}
    errs = client.insert_rows_json(f"{ds}.seg_run_log", [row])
    if errs: print("⚠ no se pudo escribir seg_run_log:", errs)

def run_step(step):
    sql = render(Path(sql_dir, f"{step}.sql").read_text())
    if dry_run:
        try:
            job = client.query(sql, job_config=cfg(True, sql))
            gb = (job.total_bytes_processed or 0) / 1e9
            print(f"  [dry-run] {step}: ~{gb:,.2f} GB (estimación; puede ser parcial en scripts)")
        except Exception as e:
            print(f"  [dry-run] {step}: no estimable ({str(e)[:160]})")
        return
    t0 = time.time()
    try:
        job = client.query(sql, job_config=cfg(False, sql))
        rows = list(job.result())
        m = dict(rows[0].items()) if rows else {}
        secs = time.time() - t0
        ok, msg = GATES[step](m)
        gb = (job.total_bytes_billed or 0) / 1e9
        print(f"  {'✓' if ok else '✗'} {step}: {msg} | {gb:,.2f} GB | {secs:,.0f}s")
        log(step, "OK" if ok else "GATE_FAILED", job.total_bytes_billed, secs, m)
        RESULTS.append({"step": step, "gb": gb, "secs": secs, **m})
        if not ok:
            raise RuntimeError(f"Compuerta de calidad no superada en {step}: {msg}")
    except RuntimeError:
        raise
    except Exception as e:
        log(step, "ERROR", None, time.time() - t0, None, str(e)[:1000])
        raise

In [5]:
# DDL idempotente (tablas, log, función). Siempre se corre; no consume datos.
client.query(render(Path(sql_dir, "00_ddl.sql").read_text())).result()

# Precondiciones de NEXT_MONTH / REASSIGN
if modo != "FULL":
    checks = [("seg_umbrales", VALUES["p_umb"], "umbrales")]
    if fuente_prev != "oficial":      # con la oficial, el paso 00c carga la previa y su compuerta la valida
        checks.insert(0, ("seg_segmentacion", VALUES["p_prev"], "segmentación previa"))
    for tabla, corte, nombre in checks:
        n = n_filas(tabla, corte)
        if n == 0:
            raise RuntimeError(f"No hay {nombre} para fecha_corte={corte} en {tabla}")
        print(f"  ✓ {nombre} {corte}: {n:,} filas")

  ✓ umbrales 2026-01-01: 231 filas


In [6]:
# ── Ejecución ──
print(f"run_id={RUN_ID}")
for step in STEPS:
    run_step(step)
print("\nPipeline completado" if not dry_run else "\nDry-run completado")

run_id=33e7664ebb69
  ✓ 00b_snapshot_estrategica: golosineros=12,668 para 2026-09-01 | 0.11 GB | 6s


C:\Users\asucasacav\AppData\Local\Temp\ipykernel_26000\2863151771.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = {"run_id": RUN_ID, "ts": dt.datetime.utcnow().isoformat(), "modo": modo, "p_corte": p_corte,


  ✓ 01_base_cliente_mes: pct_match_vb=0.954 (mín 0.8), n_clientes=198,040, dex_con_varios_sap=35, sap_distinto_dim=986 | 9.51 GB | 28s
  ✓ 02_resumen_cliente: n_clientes=198,040, sin_asignar=13,507, sin_sap=0 | 0.16 GB | 8s
  ✓ 00c_cargar_previa_oficial: previa oficial: clientes=217,052, por_definir=19,910, diamante=213, dex_con_varios_segmentos=0 | 0.04 GB | 6s
  ✓ 03_asignacion_incremental: n_filas=218,495, sin_segmento=0, filas_prev=217052, diamante=213, sin_asignar=16521 | 0.07 GB | 7s
  ✓ 05_export_tabla_alterna: filas=236,861, sin_segmentacion_interna=11,571, difiere_de_base=225,290, por_definir=16,438, seg_fuera_de_base=534, diamante base/final=0/0 (en filas de la estrategica: 0), filas_completadas_con_oficial=0, filas_boca_derivada=2,196, sin_segmento_final=11,867, diamante_base_pisa_nuestro=0, nuestro_fuera_de_lista=296 | 0.19 GB | 12s

Pipeline completado


In [7]:
# ── Resumen de la corrida ──
if RESULTS:
    display(pd.DataFrame(RESULTS))
    q = f"""SELECT des_segmentacion_transaccional AS segmento, origen, COUNT(*) n
            FROM `{ds}.seg_segmentacion` WHERE fecha_corte = @c GROUP BY 1, 2 ORDER BY 1, 2"""
    display(client.query(q, job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("c", "DATE", VALUES["p_corte"])])).to_dataframe())

,step,gb,secs,n_golosineros,n_ratios,vb_out,vb_base,n_clientes,n_dex_multi_sap,n_dex_sap_dif_dim,...,n_difiere_de_base,n_diamante_base,n_diamante_base_estrategica,n_filas_de_oficial,n_filas_derivadas,n_diamante_final,n_sin_segmento_final,n_diamante_base_pisa_nuestro,n_diamante_nuestro_fuera_de_lista,n_seg_fuera_de_base
0,00b_snapshot_estrategica,0.114295,6.257399,12668.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01_base_cliente_mes,9.508487,27.768138,NaN,7576.0,1661428970.8,1.584642e+09,198040.0,35.0,986.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02_resumen_cliente,0.164626,8.270956,NaN,NaN,NaN,NaN,198040.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00c_cargar_previa_oficial,0.041943,6.400811,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,03_asignacion_incremental,0.073400,7.459447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,05_export_tabla_alterna,0.193987,12.491684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,225290.0,0.0,0.0,0.0,2196.0,0.0,11867.0,0.0,296.0,534.0


C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,segmento,origen,n
0,0 Diamante,Previo,200
1,0 Diamante,Sin datos en ventana,13
2,1 Oro+,Previo,5703
3,1 Oro+,Reevaluado,11
4,1 Oro+,Sin datos en ventana,306
5,2 Oro,Nuevo,8
6,2 Oro,Previo,15496
7,2 Oro,Reevaluado,61
8,2 Oro,Sin datos en ventana,745
9,3 Plata,Nuevo,18


In [ ]:
# ── QA de cambios de boca (paso 06): CMP / AS / TOTAL ──
if not dry_run and "01b_cambios_boca" in STEPS:
    sql_qa = render(Path(sql_dir, "06_qa_cambios_boca.sql").read_text())
    display(client.query(sql_qa, job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("p_corte", "DATE", VALUES["p_corte"])])).to_dataframe())
    q = f"""SELECT tipo_cambio, univ_ant, univ_act, COUNT(*) n FROM `{ds}.seg_cambios_boca`
            WHERE fecha_corte = @c AND tipo_cambio NOT IN ('SIN_CAMBIO') GROUP BY 1, 2, 3 ORDER BY n DESC"""
    display(client.query(q, job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("c", "DATE", VALUES["p_corte"])])).to_dataframe())